# 05 — Report and threshold review

This notebook reads a completed validation run. It does not recompute metrics or tune on test. Thresholds are reviewed on validation and frozen only when a held-out test run is authorized.


In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import hashlib
import json
from datetime import UTC, datetime

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

REPOSITORY_ROOT = Path.cwd().resolve()
while not (REPOSITORY_ROOT / 'models' / 'referee' / 'recognition').is_dir():
    REPOSITORY_ROOT = REPOSITORY_ROOT.parent
MODEL_PROJECT = REPOSITORY_ROOT / 'models' / 'referee'


## Results location

Point this at the full `train`-profile validation output from notebook 04. Do not freeze from the `sample` wiring profile.


In [ ]:
PROFILE = 'train'
OUTPUT_ROOT = MODEL_PROJECT / 'checkpoints' / 'referee-evaluation' / PROFILE / 'validation-v2'
SUMMARY_PATH = OUTPUT_ROOT / 'summary.json'
REPORT_PATH = OUTPUT_ROOT / 'report.md'
print('Summary:', SUMMARY_PATH)


## Threshold table

The current train-profile labels have positives but no known negatives, so the apparent known-label F1 is not a meaningful threshold choice. Keep the predeclared 0.5 reference threshold for the held-out diagnostic; do not select thresholds from test results.


In [ ]:
if not SUMMARY_PATH.is_file():
    print('No summary yet. Run notebook 04 first.')
else:
    summary = json.loads(SUMMARY_PATH.read_text(encoding='utf-8'))
    for name in ('text', 'number'):
        channel = summary['channels'][name]
        points = channel.get('thresholds') or []
        if points:
            table = pd.DataFrame(points)[['threshold', 'precision', 'recall', 'f1', 'falsePositiveRate']]
            print(name)
            display(table)
        else:
            print(name, 'has no known-label threshold points.')


## AP by slice

Encoded-proxy AP, known-label AP and reviewed-label AP answer different questions. Do not replace one with another.


In [ ]:
if SUMMARY_PATH.is_file():
    rows = []
    for name in ('text', 'number', 'symbol', 'circle'):
        channel = summary['channels'][name]
        rows.append({'channel': name, 'encoded_ap': channel['encodedAp'], 'known_ap': channel.get('knownAp'), 'reviewed_ap': channel.get('reviewedAp'), 'proxy_ap': channel.get('proxyAp')})
    frame = pd.DataFrame(rows).set_index('channel')
    display(frame)
    frame[['encoded_ap', 'known_ap', 'reviewed_ap', 'proxy_ap']].plot.bar(figsize=(9, 4))
    plt.title('Referee validation AP by slice')
    plt.tight_layout()
    plt.show()
else:
    print('No summary yet.')


## Optional freeze record

Only write this after reviewing the validation report. It pins the checkpoint hash, preprocessing and chosen thresholds for a later held-out test run. Do not change it after seeing test results.


In [ ]:
WRITE_FREEZE_RECORD = False
FROZEN_THRESHOLDS = {'text': 0.5, 'number': 0.5}
FREEZE_OUTPUT = OUTPUT_ROOT / 'freeze.json'

if not WRITE_FREEZE_RECORD:
    print('Freeze record disabled.')
elif not SUMMARY_PATH.is_file():
    raise FileNotFoundError('Run validation before writing a freeze record.')
else:
    summary = json.loads(SUMMARY_PATH.read_text(encoding='utf-8'))
    if summary.get('split') != 'validation':
        raise ValueError('A freeze record must come from validation.')
    if summary['preprocessing'] == 'auto':
        raise ValueError('Resolve preprocessing before freezing.')
    prepared_summary = MODEL_PROJECT / 'data' / '.local' / 'data' / PROFILE / 'dataset-summary.json'
    if not prepared_summary.is_file():
        raise FileNotFoundError(prepared_summary)
    if any(not 0 <= value <= 1 for value in FROZEN_THRESHOLDS.values()):
        raise ValueError('Thresholds must be in [0, 1].')
    payload = {
        'schema': 'referee-evaluation-freeze-v1',
        'createdUtc': datetime.now(UTC).isoformat(),
        'checkpointSha256': summary['checkpoint']['sha256'],
        'preparedSummarySha256': hashlib.sha256(prepared_summary.read_bytes()).hexdigest(),
        'preprocessing': summary['preprocessing'],
        'thresholds': FROZEN_THRESHOLDS,
        'validationReportSha256': hashlib.sha256(REPORT_PATH.read_bytes()).hexdigest(),
    }
    FREEZE_OUTPUT.write_text(json.dumps(payload, indent=2) + '\n', encoding='utf-8')
    print('Wrote', FREEZE_OUTPUT)


## Report and limits

Read the generated report for the checkpoint score caveat, coverage and per-slice support.


In [ ]:
if REPORT_PATH.is_file():
    print(REPORT_PATH.read_text(encoding='utf-8'))
else:
    print('No report yet; run notebook 04 first.')


## Limits

- The filename score is a training-validation metric, not reviewed-label accuracy.
- Encoded bits are proxies; reviewed labels are a separate slice.
- No serving, replay timing, persistence policy or gameplay balance is validated here.
